# Visualización de Datos – ICFES Saber 11

Este notebook descarga el dataset desde Kaggle y genera visualizaciones exploratorias enfocadas en equidad: brechas por área (urbano/rural), estrato, género, tipo de colegio y departamento.

## 0. Dependencias

In [ ]:
%pip install -q "kagglehub[pandas-datasets]" pandas matplotlib seaborn

## 1. Descarga y carga desde Kaggle

Requiere credenciales en `~/.kaggle/kaggle.json`.  
Obtenerlas en **https://www.kaggle.com/settings → API → Create New Token**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../bases_datos/v1_todos_los_datos.csv")
df = pd.read_csv(DATA_PATH, sep=";", low_memory=False)

print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head(3)

## 2. Preparación de datos

In [ ]:
from pathlib import Path

PUNTAJES = ["punt_global", "punt_matematicas", "punt_lectura_critica",
            "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles"]

for col in PUNTAJES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["periodo"] = df["periodo"].astype(str)
df["anio"] = df["periodo"].str[:4].astype(int)
df["semestre"] = df["periodo"].str[4:]

df["cole_area_ubicacion"] = df["cole_area_ubicacion"].str.strip().str.title()
df["cole_naturaleza"]     = df["cole_naturaleza"].str.strip().str.title()
df["estu_genero"]         = df["estu_genero"].str.strip().str.upper()

print("Periodos:", sorted(df["periodo"].unique()))
print("Área:    ", df["cole_area_ubicacion"].value_counts().to_dict())
print("Naturaleza:", df["cole_naturaleza"].value_counts().to_dict())

In [ ]:
PUNTAJES = ["punt_global", "punt_matematicas", "punt_lectura_critica",
            "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles"]

for col in PUNTAJES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["periodo"] = df["periodo"].astype(str)
df["anio"] = df["periodo"].str[:4].astype(int)
df["semestre"] = df["periodo"].str[4:]

# Limpiar etiquetas comunes
df["cole_area_ubicacion"] = df["cole_area_ubicacion"].str.strip().str.title()
df["cole_naturaleza"]     = df["cole_naturaleza"].str.strip().str.title()
df["estu_genero"]         = df["estu_genero"].str.strip().str.upper()

print("Periodos:", sorted(df["periodo"].unique()))
print("Área:    ", df["cole_area_ubicacion"].value_counts().to_dict())
print("Naturaleza:", df["cole_naturaleza"].value_counts().to_dict())

## 3. Configuración visual

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE_AREA  = {"Urbano": "#2196F3", "Rural": "#4CAF50"}
PALETTE_NAT   = {"Oficial": "#FF5722", "No Oficial": "#9C27B0"}
PALETTE_GEN   = {"M": "#1976D2", "F": "#E91E63"}
FIG_W, FIG_H  = 14, 5

## 4. Distribución del puntaje global

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Histograma general
ax = axes[0]
df["punt_global"].dropna().plot.hist(bins=60, color="#546E7A", edgecolor="white", ax=ax)
ax.axvline(df["punt_global"].mean(), color="#F44336", lw=2, label=f'Media: {df["punt_global"].mean():.1f}')
ax.set_title("Distribución del puntaje global")
ax.set_xlabel("Puntaje global")
ax.set_ylabel("Estudiantes")
ax.legend()

# KDE por área
ax2 = axes[1]
for area, color in PALETTE_AREA.items():
    subset = df[df["cole_area_ubicacion"] == area]["punt_global"].dropna()
    if len(subset) > 0:
        subset.plot.kde(ax=ax2, color=color, lw=2.5, label=f"{area} (n={len(subset):,})")
ax2.set_title("Densidad del puntaje global – Urbano vs Rural")
ax2.set_xlabel("Puntaje global")
ax2.legend()

plt.tight_layout()
plt.savefig("../documentos_soporte/fig_distribucion_puntaje.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Evolución temporal del puntaje global

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Promedio por año
trend = df.groupby("anio")["punt_global"].mean().reset_index()
ax = axes[0]
ax.plot(trend["anio"], trend["punt_global"], marker="o", color="#1565C0", lw=2.5)
ax.set_title("Puntaje global promedio por año")
ax.set_xlabel("Año")
ax.set_ylabel("Puntaje promedio")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Promedio por año y área
trend_area = df.groupby(["anio", "cole_area_ubicacion"])["punt_global"].mean().reset_index()
ax2 = axes[1]
for area, color in PALETTE_AREA.items():
    sub = trend_area[trend_area["cole_area_ubicacion"] == area]
    ax2.plot(sub["anio"], sub["punt_global"], marker="o", color=color, lw=2.5, label=area)
ax2.set_title("Tendencia puntaje global – Urbano vs Rural")
ax2.set_xlabel("Año")
ax2.set_ylabel("Puntaje promedio")
ax2.legend()
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("../documentos_soporte/fig_tendencia_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Puntajes por área – todas las materias

In [ ]:
materias = [c for c in PUNTAJES if c != "punt_global"]
labels   = [c.replace("punt_", "").replace("_", " ").title() for c in materias]

medias = (
    df[df["cole_area_ubicacion"].isin(["Urbano", "Rural"])]
    .groupby("cole_area_ubicacion")[materias]
    .mean()
)

ax = medias.T.plot.bar(
    figsize=(10, 5), color=[PALETTE_AREA["Urbano"], PALETTE_AREA["Rural"]],
    edgecolor="white", width=0.65
)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_title("Puntaje promedio por materia – Urbano vs Rural")
ax.set_ylabel("Puntaje promedio")
ax.legend(title="Área")
plt.tight_layout()
plt.savefig("../documentos_soporte/fig_puntaje_por_materia_area.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Brecha urbano-rural por año

In [ ]:
pivot = (
    df[df["cole_area_ubicacion"].isin(["Urbano", "Rural"])]
    .groupby(["anio", "cole_area_ubicacion"])["punt_global"]
    .mean()
    .unstack()
)
pivot["brecha"] = pivot["Urbano"] - pivot["Rural"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(pivot.index, pivot["brecha"], color="#FF7043", edgecolor="white")
ax.axhline(pivot["brecha"].mean(), color="#B71C1C", ls="--", lw=1.5,
           label=f'Media brecha: {pivot["brecha"].mean():.1f} pts')
ax.set_title("Brecha de puntaje global Urbano − Rural por año")
ax.set_xlabel("Año")
ax.set_ylabel("Diferencia de puntos")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig("../documentos_soporte/fig_brecha_urbano_rural.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Puntaje global por estrato socioeconómico

In [ ]:
orden_estrato = ["Sin Estrato", "Estrato 1", "Estrato 2", "Estrato 3",
                 "Estrato 4", "Estrato 5", "Estrato 6"]

df["fami_estratovivienda"] = df["fami_estratovivienda"].str.strip().str.title()
estrato_validos = df[df["fami_estratovivienda"].isin(orden_estrato)]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=estrato_validos, x="fami_estratovivienda", y="punt_global",
    order=orden_estrato, palette="Blues", flierprops=dict(marker=".", alpha=0.3), ax=ax
)
ax.set_title("Distribución del puntaje global por estrato")
ax.set_xlabel("Estrato de vivienda")
ax.set_ylabel("Puntaje global")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig("../documentos_soporte/fig_puntaje_por_estrato.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Puntaje global por tipo de colegio (oficial vs no oficial)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Boxplot naturaleza × área
ax = axes[0]
orden_nat = ["Oficial", "No Oficial"]
sns.boxplot(
    data=df[df["cole_naturaleza"].isin(orden_nat)],
    x="cole_naturaleza", y="punt_global", hue="cole_area_ubicacion",
    order=orden_nat, palette=PALETTE_AREA,
    flierprops=dict(marker=".", alpha=0.2), ax=ax
)
ax.set_title("Puntaje global: tipo de colegio × área")
ax.set_xlabel("Naturaleza del colegio")
ax.set_ylabel("Puntaje global")

# Tendencia por naturaleza
trend_nat = df.groupby(["anio", "cole_naturaleza"])["punt_global"].mean().reset_index()
ax2 = axes[1]
for nat, color in PALETTE_NAT.items():
    sub = trend_nat[trend_nat["cole_naturaleza"] == nat]
    ax2.plot(sub["anio"], sub["punt_global"], marker="o", color=color, lw=2.5, label=nat)
ax2.set_title("Tendencia puntaje – Oficial vs No Oficial")
ax2.set_xlabel("Año")
ax2.set_ylabel("Puntaje promedio")
ax2.legend()
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("../documentos_soporte/fig_colegio_oficial.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Brecha de género por materia

In [ ]:
genero_valido = df[df["estu_genero"].isin(["M", "F"])]

# Brecha promedio por materia
medias_gen = genero_valido.groupby("estu_genero")[materias].mean()
brecha_gen = (medias_gen.loc["M"] - medias_gen.loc["F"]).rename("brecha_M_F")

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

ax = axes[0]
colors = ["#1976D2" if v >= 0 else "#E91E63" for v in brecha_gen]
brecha_gen.plot.barh(ax=ax, color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.set_yticklabels(labels, fontsize=10)
ax.set_title("Brecha de género por materia (M − F)")
ax.set_xlabel("Diferencia de puntos (azul = hombres arriba)")

# Tendencia puntaje global por género
trend_gen = genero_valido.groupby(["anio", "estu_genero"])["punt_global"].mean().reset_index()
ax2 = axes[1]
for gen, color in PALETTE_GEN.items():
    sub = trend_gen[trend_gen["estu_genero"] == gen]
    lbl = "Masculino" if gen == "M" else "Femenino"
    ax2.plot(sub["anio"], sub["punt_global"], marker="o", color=color, lw=2.5, label=lbl)
ax2.set_title("Tendencia puntaje global por género")
ax2.set_xlabel("Año")
ax2.set_ylabel("Puntaje promedio")
ax2.legend()
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("../documentos_soporte/fig_brecha_genero.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Acceso a tecnología y puntaje

In [ ]:
tech_cols = {
    "fami_tienecomputador": "Tiene computador",
    "fami_tieneinternet":   "Tiene internet",
    "fami_tienelavadora":   "Tiene lavadora",
    "fami_tieneautomovil":  "Tiene automóvil",
}

fig, axes = plt.subplots(1, len(tech_cols), figsize=(16, 5), sharey=True)

for ax, (col, titulo) in zip(axes, tech_cols.items()):
    sub = df[[col, "punt_global"]].dropna()
    sub[col] = sub[col].str.strip().str.title()
    orden = sorted(sub[col].unique())
    sns.boxplot(data=sub, x=col, y="punt_global", order=orden,
                palette="Set2", flierprops=dict(marker=".", alpha=0.2), ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("")
    if ax == axes[0]:
        ax.set_ylabel("Puntaje global")
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Puntaje global según bienes del hogar", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../documentos_soporte/fig_tecnologia_hogar.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Mapa de calor: promedio por departamento

In [ ]:
depto_media = (
    df.groupby("cole_depto_ubicacion")[materias + ["punt_global"]]
    .mean()
    .sort_values("punt_global", ascending=False)
    .head(30)  # top 30 para legibilidad
)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    depto_media[materias],
    annot=True, fmt=".0f", cmap="RdYlGn",
    linewidths=0.4, linecolor="white",
    yticklabels=depto_media.index, ax=ax
)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_title("Puntaje promedio por materia y departamento (top 30)")
plt.tight_layout()
plt.savefig("../documentos_soporte/fig_heatmap_departamentos.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Educación de padres y puntaje global

In [ ]:
orden_edu = [
    "Ninguno",
    "Primaria Incompleta", "Primaria Completa",
    "Secundaria (Bachillerato) Incompleta", "Secundaria (Bachillerato) Completa",
    "Tecnica O Tecnologica Incompleta", "Tecnica O Tecnologica Completa",
    "Educacion Profesional Incompleta", "Educacion Profesional Completa",
    "Postgrado",
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, titulo in zip(
    axes,
    ["fami_educacionmadre", "fami_educacionpadre"],
    ["Educación madre", "Educación padre"]
):
    sub = df[[col, "punt_global"]].dropna()
    sub[col] = sub[col].str.strip().str.title()
    presentes = [o for o in orden_edu if o in sub[col].unique()]
    medias_edu = sub.groupby(col)["punt_global"].mean().reindex(presentes)
    medias_edu.plot.barh(ax=ax, color="#5C6BC0", edgecolor="white")
    ax.set_title(f"Puntaje global por {titulo}")
    ax.set_xlabel("Puntaje promedio")
    ax.set_ylabel("")

plt.tight_layout()
plt.savefig("../documentos_soporte/fig_educacion_padres.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Resumen estadístico de brechas

In [ ]:
resumen = {}

# Brecha urbano-rural
for area in ["Urbano", "Rural"]:
    resumen[f"Prom. {area}"] = df[df["cole_area_ubicacion"] == area]["punt_global"].mean()
resumen["Brecha Urbano-Rural"] = resumen["Prom. Urbano"] - resumen["Prom. Rural"]

# Brecha oficial-no oficial
for nat in ["Oficial", "No Oficial"]:
    resumen[f"Prom. {nat}"] = df[df["cole_naturaleza"] == nat]["punt_global"].mean()
resumen["Brecha Oficial/No Oficial"] = resumen["Prom. No Oficial"] - resumen["Prom. Oficial"]

# Brecha de género
for gen, label in [("M", "Hombres"), ("F", "Mujeres")]:
    resumen[f"Prom. {label}"] = df[df["estu_genero"] == gen]["punt_global"].mean()
resumen["Brecha Género (M-F)"] = resumen["Prom. Hombres"] - resumen["Prom. Mujeres"]

# Brecha estrato extremo
prom_e1 = df[df["fami_estratovivienda"] == "Estrato 1"]["punt_global"].mean()
prom_e6 = df[df["fami_estratovivienda"] == "Estrato 6"]["punt_global"].mean()
resumen["Prom. Estrato 1"] = prom_e1
resumen["Prom. Estrato 6"] = prom_e6
resumen["Brecha Estrato 6-1"] = prom_e6 - prom_e1

resumen_df = pd.Series(resumen).rename("Valor").to_frame()
resumen_df["Valor"] = resumen_df["Valor"].round(2)
resumen_df.style.background_gradient(cmap="RdYlGn", subset="Valor")